## Fantasy Football Playoff Probability Simulation

This notebook calculates the playoff probabilities for a 10-team fantasy football league. It uses the final results from Week 10 to establish a new baseline and then simulates the remainder of the regular season (Weeks 11-14) thousands of times to determine the odds for each team making the playoffs (top 4).

### Step 1: Initial Setup and Data Import

First, we import the necessary libraries and define the league's data as of the beginning of Week 10. This includes the list of teams, their records and points for, and the final scores from the Week 10 matchups.

In [ ]:
import numpy as np
import random
import copy
from numpy.random import normal as npnormal

# The list of all teams in the league
# Note: 'Didnt Start Dart. Shart' has been updated to 'Jonathan Taylor Day'
teams = [
    'Jonathan Taylor Day', 'Hope Mahomes Still Standing', 'Lizard lizard lizard',
    'MR. SNIFFLES', 'Why did I trade JSN?', 'No Mo Toe Joe',
    'In My Football Era', 'Kittle Me This', 'Rippin Darts', 'Erica Loves Sports'
]

# Standings *before* Week 10 games were played
standings_before_week_10 = {
    'Jonathan Taylor Day': [7, 1110.22], 
    'Hope Mahomes Still Standing': [6, 1045.86],
    'Lizard lizard lizard': [5, 1037.84],
    'MR. SNIFFLES': [5, 1016.48],
    'Why did I trade JSN?': [4, 1011.18],
    'No Mo Toe Joe': [4, 951.24],
    'In My Football Era': [4, 919.34],
    'Kittle Me This': [4, 855.36],
    'Rippin Darts': [3, 928.14],
    'Erica Loves Sports': [3, 858.74]
}

# The final scores from the Week 10 matchups
week_10_results = {
    'Jonathan Taylor Day': 119.90, 'Kittle Me This': 112.62,
    'No Mo Toe Joe': 118.86, 'Lizard lizard lizard': 102.44,
    'Why did I trade JSN?': 101.50, 'Rippin Darts': 101.10,
    'Erica Loves Sports': 121.50, 'In My Football Era': 115.24,
    'MR. SNIFFLES': 91.72, 'Hope Mahomes Still Standing': 116.92
}

# Week 10 matchups for determining winner/loser
week_10_matchups = [
    {'Jonathan Taylor Day', 'Kittle Me This'},
    {'No Mo Toe Joe', 'Lizard lizard lizard'},
    {'Why did I trade JSN?', 'Rippin Darts'},
    {'Erica Loves Sports', 'In My Football Era'},
    {'MR. SNIFFLES', 'Hope Mahomes Still Standing'}
]

### Step 2: Calculate New Standings After Week 10

Using the Week 10 results, we'll now calculate the new, official standings. This will be the starting point for our simulation of the rest of the season.

In [ ]:
standings_after_week_10 = {}

for matchup in week_10_matchups:
    team1, team2 = list(matchup)
    team1_score = week_10_results[team1]
    team2_score = week_10_results[team2]
    
    # Determine winner and loser
    if team1_score > team2_score:
        winner, loser = team1, team2
    else:
        winner, loser = team2, team1
        
    # Update winner's record
    winner_old_stats = standings_before_week_10[winner]
    standings_after_week_10[winner] = [
        winner_old_stats[0] + 1, # Add a win
        winner_old_stats[1] + week_10_results[winner]
    ]
    
    # Update loser's record
    loser_old_stats = standings_before_week_10[loser]
    standings_after_week_10[loser] = [
        loser_old_stats[0], # Wins stay the same
        loser_old_stats[1] + week_10_results[loser]
    ]

# Print the new standings to verify correctness
print("--- Standings After Week 10 Results ---")
sorted_standings = sorted(standings_after_week_10.items(), key=lambda item: (item[1][0], item[1][1]), reverse=True)

for team, stats in sorted_standings:
    print(f"{team:<30} | Wins: {stats[0]}, Points: {stats[1]:.2f}")

### Step 3: Define Simulation Functions

These functions will form the core of our simulation. 
- `simulate_game`: Simulates a single game between two teams with randomized scores.
- `get_playoff_teams`: Ranks all teams by wins (then points for) and returns the top 4.

In [ ]:
def simulate_game(team1_name, team2_name, current_standings):
    """Simulates a game and returns the updated stats for both teams."""
    # Scores are drawn from a normal distribution (mean=108, std_dev=25)
    team1_score = npnormal(108, 25)
    team2_score = npnormal(108, 25)

    if team1_score > team2_score:
        winner_name, loser_name = team1_name, team2_name
        winner_score, loser_score = team1_score, team2_score
    else: # team2 wins in a tie
        winner_name, loser_name = team2_name, team1_name
        winner_score, loser_score = team2_score, team1_score

    winner_stats = current_standings[winner_name]
    loser_stats = current_standings[loser_name]

    return {
        winner_name: [winner_stats[0] + 1, winner_stats[1] + winner_score],
        loser_name: [loser_stats[0], loser_stats[1] + loser_score]
    }

def get_playoff_teams(final_standings):
    """Determines the top 4 teams based on wins, then points for."""
    # First, sort by points for as the tiebreaker
    standings = dict(sorted(final_standings.items(), key=lambda item: item[1][1], reverse=True))
    # Then, sort by wins as the primary ranking factor
    standings = dict(sorted(standings.items(), key=lambda item: item[1][0], reverse=True))
    # Return the top 4 teams
    return list(standings.keys())[:4]

### Step 4: Run the Main Simulation

Now we'll loop through the simulation 100,000 times. In each loop, we simulate Weeks 11 through 14 with random matchups and scores. We then record which four teams made the playoffs for that specific simulation run.

In [ ]:
playoff_counts = {team: 0 for team in teams}
num_simulations = 100000

print(f"Running {num_simulations} simulations from Week 11 to 14...")

for _ in range(num_simulations):
    # Start each simulation with a fresh copy of the post-Week 10 standings
    current_season_data = copy.deepcopy(standings_after_week_10)

    # Simulate the rest of the season (Weeks 11, 12, 13, 14)
    for week in range(11, 15):
        next_week_standings = {}
        # Randomize matchups for the week since the schedule is unknown
        shuffled_teams = list(current_season_data.keys())
        random.shuffle(shuffled_teams)
        
        for i in range(0, len(shuffled_teams), 2):
            team1, team2 = shuffled_teams[i], shuffled_teams[i+1]
            game_results = simulate_game(team1, team2, current_season_data)
            next_week_standings.update(game_results)
        
        current_season_data = next_week_standings

    # After simulating the season, find the playoff teams
    final_playoff_teams = get_playoff_teams(current_season_data)
    for team in final_playoff_teams:
        playoff_counts[team] += 1

print("Simulation complete.")

### Step 5: Display the Final Results

Finally, we calculate the percentage chance for each team to make the playoffs based on how many times they finished in the top 4 across all simulations. The results are sorted from highest to lowest probability.

In [ ]:
print("\n--- Updated Playoff Probabilities (After Week 10) ---")

# Sort teams by playoff probability for the final display
sorted_results = sorted(playoff_counts.items(), key=lambda item: item[1], reverse=True)

print(f"{'Rank':<5} | {'Team':<30} | {'Record':<12} | {'Playoff Odds'}")
print("-" * 70)

rank = 1
for team, count in sorted_results:
    percentage = (count / num_simulations) * 100
    record = standings_after_week_10[team]
    record_str = f"{record[0]}-{10-record[0]}-0"
    print(f"{rank:<5} | {team:<30} | {record_str:<12} | {percentage:.2f}%")
    rank += 1